# User behavior and temporal trends

This notebook is the shared handoff surface for user activity, rating-year trends, release-year trends, decade comparison, and Challenge Question 2. It imports the leader's definitions instead of rebuilding them locally.

In [ ]:
from pathlib import Path
import os
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from load_data import load_movielens
from preprocess import build_shared_tables
from metrics import activity_group_summary, decade_summary, ratings_by_year, release_year_stats, user_activity_summary

data_dir = Path(os.environ.get('MOVIELENS_DATA_DIR', PROJECT_ROOT / 'data' / 'ml-20m'))
try:
    frames = load_movielens(data_dir)
except FileNotFoundError:
    frames = load_movielens(PROJECT_ROOT / 'data' / 'sample')
tables = build_shared_tables(frames['ratings'], frames['movies'], frames.get('tags'))
ratings = tables['ratings']
movies = tables['movies']
user_stats = tables['user_stats']
exploded_genres = tables['exploded_genres']

## User activity distribution and Challenge Question 2

In [ ]:
display(user_activity_summary(user_stats))
display(user_stats.nlargest(20, 'rating_count'))
display(activity_group_summary(user_stats))
print('Use rating_count for activity. Compare top users descriptively with percentile-based less-active groups.')

## Rating-year, release-year, and decade views

In [ ]:
rating_year_stats = ratings_by_year(ratings)
release_year_view = release_year_stats(ratings, movies, min_rating_count=1000)
decade_view = decade_summary(ratings, movies, exploded_genres)
display(rating_year_stats)
display(release_year_view)
display(decade_view)
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'summary_tables'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
rating_year_stats.to_csv(OUTPUT_DIR / 'user_temporal_rating_year.csv', index=False)
release_year_view.to_csv(OUTPUT_DIR / 'user_temporal_release_year.csv', index=False)
decade_view.to_csv(OUTPUT_DIR / 'user_temporal_decade.csv', index=False)

Interpretation must distinguish rating year from release year, mention incomplete support in the final rating year, and avoid causal claims about newer or older movies.